# S3.3 — Schema Definition — StructType, StructField
**Date completed:** September 2026  
**Status:** Completed  
**Key concept:** Always define schema explicitly in production

## Explicit schema definition:

In [0]:
# ============================================================
# Cell 3 — Explicit Schema Definition
#
# Goal:
# 1. Define schema using StructType and StructField
# 2. Read customers CSV with correct types
# 3. Compare inferSchema vs explicit schema
# 4. Prove explicit schema is always better
# ============================================================


from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType)

import pyspark.sql.functions as F

# ------------------------------------------------------------
# Step 1 — Define explicit schema for customers CSV
# Each StructField: column_name, data_type, nullable
# ------------------------------------------------------------
customers_schema = StructType([
    StructField("customer_id",      IntegerType(),   False),  # cannot be null
    StructField("tax_id",           DoubleType(),    True),
    StructField("tax_code",         StringType(),    True),
    StructField("customer_name",    StringType(),    True),
    StructField("state",            StringType(),    True),
    StructField("city",             StringType(),    True),
    StructField("postcode",         StringType(),    True),
    StructField("street",           StringType(),    True),
    StructField("number",           StringType(),    True),
    StructField("unit",             StringType(),    True),
    StructField("region",           StringType(),    True),
    StructField("district",         StringType(),    True),
    StructField("lon",              DoubleType(),    True),
    StructField("lat",              DoubleType(),    True),
    StructField("ship_to_address",  StringType(),    True),
    StructField("valid_from",       LongType(),      True),   # Unix timestamp
    StructField("valid_to",         LongType(),      True),   # Unix timestamp
    StructField("units_purchased",  DoubleType(),    True),
    StructField("loyalty_segment",  IntegerType(),   True),
])

# ------------------------------------------------------------
# Step 2 — Read CSV with explicit schema
# No inferSchema needed — reads file ONCE
# ------------------------------------------------------------
df_customers = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE")  \
    .schema(customers_schema) \
    .csv("dbfs:/databricks-datasets/retail-org/customers/customers.csv")

print(f"Row count : {df_customers.count()}")
print()

# ------------------------------------------------------------
# Step 3 — Verify schema is correct
# ------------------------------------------------------------
print("--- Schema with explicit definition ---")
df_customers.printSchema()
print()

# ============================================================
# Cell 4 — Corrected Unix timestamp conversion
#
# Goal:
# 1. Convert valid_from to readable date correctly
# 2. Show alias naming matters
# ============================================================

print("=== UNIX TIMESTAMP TO READABLE DATE ===")
print()

df_customers.select(
    "customer_id",
    "customer_name",
    "city",
    F.from_unixtime("valid_from").cast("timestamp").alias("valid_from_date"),
    F.from_unixtime("valid_to").cast("timestamp").alias("valid_to_date"),
    "loyalty_segment"
).show(5, truncate=30)

print()
print("valid_from = customer record start date")
print("valid_to   = customer record expiry date (NULL = still active)")
print("Unix timestamp correctly stored as LongType")
print("from_unixtime() converts to readable datetime")

In [0]:
# ============================================================
# Cell 5 — Complex Schema Types: ArrayType and MapType
#
# Goal:
# 1. Understand ArrayType — multiple values in one column.
# 2. Understand MapType — key-value pairs in one column.
# 3. Define explicit schemas using StructType and StructField.
# 4. Access individual array elements.
# 5. Access individual map values.
# 6. Understand nullable vs containsNull.
#
# Environment: Databricks Serverless
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    ArrayType,
    MapType
)

import pyspark.sql.functions as F

print("=== COMPLEX SCHEMA TYPES ===")
print()


# ============================================================
# STEP 1 — ArrayType: Define Schema
#
# ArrayType represents multiple values inside one column.
#
# Example:
# items = ["Mobile", "Charger", "Case"]
#
# nullable=True    → Entire column can be NULL.
# containsNull=True → Individual array elements can be NULL.
# ============================================================

print("=== STEP 1: ARRAYTYPE SCHEMA ===")

array_schema = StructType([

    StructField(
        "order_id",
        IntegerType(),
        False
    ),

    StructField(
        "customer",
        StringType(),
        True
    ),

    StructField(
        "items",
        ArrayType(
            StringType(),
            containsNull=True
        ),
        True
    )

])


# ============================================================
# STEP 2 — Create DataFrame with ArrayType
# ============================================================

array_data = [
    (1001, "Dhiraj", ["Mobile", "Charger", "Case"]),
    (1002, "Rahul",  ["Laptop"]),
    (1003, "Priya",  ["Shoes", "Bag"])
]

df_array = spark.createDataFrame(
    array_data,
    schema=array_schema
)

print("\n--- ArrayType Schema ---")
df_array.printSchema()

print("\n--- ArrayType Data ---")
df_array.show(truncate=False)


# ============================================================
# STEP 3: Access Array Elements Safely
#
# Goal:
# 1. Extract first and second array elements.
# 2. Avoid INVALID_ARRAY_INDEX errors.
# 3. Return NULL when an element does not exist.
# ============================================================

print("=== STEP 3: ACCESS ARRAY ELEMENTS SAFELY ===")

df_array.select(
    "order_id",
    "customer",
    "items",

    # First element — index 0
    F.get(F.col("items"), 0).alias("first_item"),

    # Second element — index 1
    # Returns NULL if the element does not exist
    F.get(F.col("items"), 1).alias("second_item")

).show(truncate=False)


# ============================================================
# STEP 4 — Count Array Elements
#
# size() returns the number of elements inside an array.
# ============================================================

print("\n=== STEP 4: COUNT ARRAY ELEMENTS ===")

df_array.select(

    "order_id",
    "customer",

    F.size("items").alias("total_items")

).show(truncate=False)


# ============================================================
# STEP 5 — MapType: Define Schema
#
# MapType stores key-value pairs.
#
# Example:
# {"brand": "Samsung", "color": "Black", "ram": "8GB"}
#
# Key Type   = StringType
# Value Type = StringType
#
# Map keys cannot be NULL.
# valueContainsNull controls whether values can be NULL.
# ============================================================

print("\n=== STEP 5: MAPTYPE SCHEMA ===")

map_schema = StructType([

    StructField(
        "product_id",
        IntegerType(),
        False
    ),

    StructField(
        "name",
        StringType(),
        True
    ),

    StructField(
        "attributes",
        MapType(
            StringType(),
            StringType(),
            valueContainsNull=True
        ),
        True
    )

])


# ============================================================
# STEP 6 — Create DataFrame with MapType
# ============================================================

map_data = [

    (
        101,
        "Mobile",
        {
            "brand": "Samsung",
            "color": "Black",
            "ram": "8GB"
        }
    ),

    (
        102,
        "Laptop",
        {
            "brand": "Dell",
            "screen": "15inch",
            "os": "Windows"
        }
    ),

    (
        103,
        "Shoes",
        {
            "brand": "Nike",
            "size": "42",
            "color": "White"
        }
    )

]

df_map = spark.createDataFrame(
    map_data,
    schema=map_schema
)

print("\n--- MapType Schema ---")
df_map.printSchema()

print("\n--- MapType Data ---")
df_map.show(truncate=False)


# ============================================================
# STEP 7 — Access Map Values
#
# Access individual values using their corresponding keys.
# ============================================================

print("\n=== STEP 7: ACCESS MAP VALUES ===")

df_map.select(

    "product_id",
    "name",

    F.col("attributes")["brand"].alias("brand"),

    F.col("attributes")["color"].alias("color")

).show(truncate=False)


# ============================================================
# STEP 8 — Extract Map Keys and Values
#
# map_keys()   → Returns available keys as an array.
# map_values() → Returns corresponding values as an array.
# ============================================================

print("\n=== STEP 8: MAP KEYS AND VALUES ===")

df_map.select(

    "product_id",
    "name",

    F.map_keys("attributes").alias("attribute_keys"),

    F.map_values("attributes").alias("attribute_values")

).show(truncate=False)


# ============================================================
# STEP 9 — Final Learning Summary
# ============================================================

print()
print("=" * 65)
print("COMPLEX SCHEMA TYPES — FINAL SUMMARY")
print("=" * 65)

print("ArrayType : Stores multiple elements in one column.")
print("MapType   : Stores key-value pairs in one column.")
print()
print("Array indexing starts from 0.")
print("Map values are accessed using their keys.")
print()
print("nullable        : Controls whether the entire column can be NULL.")
print("containsNull    : Controls whether array elements can be NULL.")
print("valueContainsNull: Controls whether map values can be NULL.")

print()
print("=== CELL 5 COMPLETED ===")

# S3.3 — Schema Definition: Key Takeaways

## 1. Why Do We Need Explicit Schemas?

A schema defines the structure of a DataFrame:

- Column names
- Data types
- Nullability
- Nested data structures

An explicit schema helps us control how Spark interprets incoming data.

### inferSchema vs Explicit Schema

| Feature | inferSchema | Explicit Schema |
|---|---|---|
| CSV schema discovery | Additional input pass | No inference pass |
| Data types | Automatically inferred | Developer-defined |
| Type accuracy | May infer unintended types | Depends on schema correctness |
| Recurring production ingestion | Less predictable | Preferred with a reliable source contract |
| Schema maintenance | Automatically inferred | Must be maintained when source changes |

**Enterprise principle:** Prefer explicit schemas when the expected source structure is known and governed.

---

## 2. StructType and StructField

StructType defines the complete schema.

StructField defines each individual field.

```python
schema = StructType([
    StructField("column_name", StringType(), True)
])
```

### Nullability

- nullable=True: The column is declared nullable.
- nullable=False: The column is declared non-nullable.

Important: Declaring nullable=False does not replace actual data-quality validation.

Our CSV experiment demonstrated that the CSV reader can return a nullable output schema even when the supplied field was declared non-nullable.

---

## 3. Important Data Types

| Data Type | Purpose |
|---|---|
| StringType() | Text, codes and identifiers |
| IntegerType() | 32-bit whole numbers |
| LongType() | 64-bit integers, including Unix timestamps stored as integer seconds |
| DoubleType() | Approximate floating-point numbers |
| DecimalType(p,s) | Exact decimal representation, useful for financial values |
| DateType() | Calendar dates |
| TimestampType() | Timestamp values |
| BooleanType() | True / False |
| ArrayType() | Collection of elements |
| MapType() | Key-value pairs |
| StructType() | Structured collection of named fields |

---

## 4. Real Practical — customers.csv

Our experiment:

- Total records: 28,813
- Explicit schema: 19 columns
- valid_from: LongType
- valid_to: LongType

We converted Unix timestamp values using:

```python
F.from_unixtime("valid_from").alias("valid_from_date")

F.from_unixtime("valid_to").alias("valid_to_date")
```

Example output:

valid_from_date = 2018-07-29 00:30:33

Important:
- from_unixtime() converts Unix seconds to formatted datetime text.
- Output depends on the session timezone.
- Unix milliseconds require different handling.
- NULL valid_to may represent an open-ended record, depending on the source-system business rules.

---

## 5. CSV Reading Modes

| Mode | Behaviour |
|---|---|
| PERMISSIVE | Attempts to retain records with malformed fields |
| DROPMALFORMED | Drops malformed records |
| FAILFAST | Stops execution when malformed data is encountered |

Example:

```python
df = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(customers_schema)
    .csv(file_path)
)
```

Parsing mode and data-quality validation are different concepts.

Malformed records should be captured and audited where required.

---

## 6. Complex Schema Types

### ArrayType — Multiple Elements in One Column

Example:

```python
["Mobile", "Charger", "Case"]
```

Schema:

```python
ArrayType(StringType(), containsNull=True)
```

Use cases:
- Order items
- Product tags
- Categories
- Nested API data

Access elements safely:

```python
F.get(F.col("items"), 0).alias("first_item")
F.get(F.col("items"), 1).alias("second_item")
```

Array indexing starts at zero.

get() returns NULL when the requested index is outside the array bounds.

This is useful in our ANSI-enabled Databricks environment.

### MapType — Key-Value Pairs

Example:

```python
{
    "brand": "Samsung",
    "color": "Black",
    "ram": "8GB"
}
```

Schema:

```python
MapType(
    StringType(),
    StringType(),
    valueContainsNull=True
)
```

Use cases:
- Product attributes
- Metadata
- Dynamic configuration
- Semi-structured data

Access values:

```python
F.col("attributes")["brand"]
```

Missing map keys return NULL when accessed using this expression.

---

## 7. Three Important Nullability Concepts

| Property | Meaning |
|---|---|
| nullable | Whether the entire column can be NULL |
| containsNull | Whether individual array elements can be NULL |
| valueContainsNull | Whether map values can be NULL |

Example:

```python
StructField(
    "items",
    ArrayType(StringType(), containsNull=False),
    True
)
```

Meaning:
- The complete items column may be NULL.
- If an array exists, its individual elements must not be NULL according to the declared schema.

---

## 8. When Do We Use Each Complex Type?

| Type | Example | Purpose |
|---|---|---|
| ArrayType | ["Mobile", "Case"] | Multiple elements |
| MapType | {"brand": "Samsung"} | Dynamic key-value attributes |
| StructType | {"name": "Rahul", "age": 30} | Structured named fields |

Complex types can be nested.

For example, an order can contain an ArrayType of StructType records, where each product contains a MapType of attributes.

Nested JSON flattening will be covered in S3.5 using functions such as explode(), explode_outer() and field selection.

---

## 9. Schema Definition vs Schema Evolution

Schema Definition:
Specifies the expected structure of incoming data.

Schema Evolution:
Handles supported structural changes when the source schema changes over time.

Example:

Initial source:
customer_id, customer_name, city

Updated source:
customer_id, customer_name, city, email

Schema evolution must be managed deliberately to avoid breaking downstream processing.

---

## Final Interview Takeaways

**Q: Why use an explicit schema?**

To provide predictable data types, avoid unnecessary schema inference and maintain a controlled source-data contract.

**Q: What is StructType?**

A structured data type containing named fields, each defined using StructField.

**Q: What is the difference between ArrayType and MapType?**

ArrayType stores collections of elements, while MapType stores key-value pairs.

**Q: Does explicit schema guarantee data quality?**

No. Schema definition and data-quality validation are separate responsibilities.

**Q: Why is schema definition important in enterprise pipelines?**

It helps maintain predictable ingestion, data-type consistency, downstream compatibility and controlled handling of source changes.

### Final Learning Principle

A schema defines what Spark expects.

Data-quality validation checks whether incoming data satisfies those expectations.

Schema evolution manages supported changes to that structure over time.